In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

# STRATEGY: DV01-neutral 2s10s Treasury flattener
DATA_FILE = Path("gsw_yields_2025.csv")
if not DATA_FILE.exists():
    raise FileNotFoundError(
        f"Could not find {DATA_FILE}. Put the CSV in the same folder as this notebook "
        "or change DATA_FILE to the correct path."
    )

df = pd.read_csv(DATA_FILE)
cols = ["Date", "BETA0", "BETA1", "BETA2", "BETA3", "TAU1", "TAU2"]
df = df[cols].copy()
df["Date"] = pd.to_datetime(df["Date"])
df = df.replace(-999.99, np.nan)

# GSW/NSS beta coefficients are reported in percentage points; convert to decimals.
for c in ["BETA0", "BETA1", "BETA2", "BETA3"]:
    df[c] = df[c] / 100.0


In [ ]:
df = (
    df.sort_values("Date")
      .set_index("Date")
      .loc["1983-12-30":"2025-09-05"]
      .resample("W-FRI").last()
      .dropna()
)

T2, T10 = 2.0, 10.0
dt = 7 / 365.0  # one week in years


def nss_yield(t, b0, b1, b2, b3, tau1, tau2):
    """Nelson-Siegel-Svensson continuously compounded zero yield."""
    x1 = t / tau1
    x2 = t / tau2
    term1 = (1 - np.exp(-x1)) / x1
    term2 = term1 - np.exp(-x1)
    term3 = (1 - np.exp(-x2)) / x2 - np.exp(-x2)
    return b0 + b1 * term1 + b2 * term2 + b3 * term3


def curve_yield(row, maturity):
    return nss_yield(
        maturity,
        row.BETA0, row.BETA1, row.BETA2, row.BETA3,
        row.TAU1, row.TAU2,
    )


# Spot curve points used for pricing and hedging.
df["y_2"] = df.apply(lambda r: curve_yield(r, T2), axis=1)
df["y_10"] = df.apply(lambda r: curve_yield(r, T10), axis=1)
df["y_1w"] = df.apply(lambda r: curve_yield(r, dt), axis=1)

# Treat each maturity as a zero-coupon bond with unit face value.
df["P2"] = np.exp(-df["y_2"] * T2)
df["P10"] = np.exp(-df["y_10"] * T10)

# Dollar-value-of-1bp per $1 face for a continuously compounded zero-coupon bond.
df["DV01_2"] = T2 * df["P2"] * 1e-4
df["DV01_10"] = T10 * df["P10"] * 1e-4

# Long 10Y / short 2Y with first-order parallel-rate DV01 neutralization.
df["hedge_ratio"] = df["DV01_10"] / df["DV01_2"]

equity_init = 1_000_000.0
margin_rate = 0.10


def y_from_prev_curve(row_prev, t_short):
    """Evaluate last week's NSS curve at a shorter maturity for roll/carry."""
    return curve_yield(row_prev, t_short)


In [ ]:
# Initialize columns used by the weekly strategy loop.
strategy_cols = [
    "q2", "q10", "cash", "equity", "ret",
    "pnl2_carry", "pnl2_yield", "pnl10_carry", "pnl10_yield",
    "pnl_carry", "pnl_yield", "pnl", "cash_interest",
    "y_2p", "y_10p", "mv2", "mv10", "gross_exposure",
    "longpos", "shortpos",
]
for c in strategy_cols:
    df[c] = np.nan

# Weekly loop.
equity = equity_init
equity_prev = equity_init
idx = df.index.to_list()

for i, t in enumerate(idx):
    r = df.loc[t]
    is_last = i == len(idx) - 1

    # Close previous week's book: position P&L (carry + yield move) + cash interest.
    if i > 0:
        tprev = idx[i - 1]
        rprev = df.loc[tprev]
        q2_prev = df.at[tprev, "q2"]
        q10_prev = df.at[tprev, "q10"]

        # Prices when positions were opened last week at full maturities.
        P2_prev = np.exp(-df.at[tprev, "y_2"] * T2)
        P10_prev = np.exp(-df.at[tprev, "y_10"] * T10)

        # Carry-only prices today: last week's curve evaluated one week shorter.
        y2_prev_roll = y_from_prev_curve(rprev, T2 - dt)
        y10_prev_roll = y_from_prev_curve(rprev, T10 - dt)
        P2_carry = np.exp(-y2_prev_roll * (T2 - dt))
        P10_carry = np.exp(-y10_prev_roll * (T10 - dt))

        # Actual prices today at the rolled maturities using today's curve.
        y2_today_roll = curve_yield(r, T2 - dt)
        y10_today_roll = curve_yield(r, T10 - dt)
        P2_today = np.exp(-y2_today_roll * (T2 - dt))
        P10_today = np.exp(-y10_today_roll * (T10 - dt))

        # P&L decomposition by leg.
        pnl2_carry = q2_prev * (P2_carry - P2_prev)
        pnl2_yield = q2_prev * (P2_today - P2_carry)
        pnl10_carry = q10_prev * (P10_carry - P10_prev)
        pnl10_yield = q10_prev * (P10_today - P10_carry)

        pnl_carry = pnl2_carry + pnl10_carry
        pnl_yield = pnl2_yield + pnl10_yield
        pnl_pos = pnl_carry + pnl_yield

        # Interest earned/paid on residual cash over the week.
        cash_prev = df.at[tprev, "cash"]
        y1w_prev = df.at[tprev, "y_1w"]
        cash_int = cash_prev * (np.exp(y1w_prev * dt) - 1.0)

        total_pnl = pnl_pos + cash_int
        equity += total_pnl
        df.at[t, "ret"] = total_pnl / equity_prev

        df.at[t, "pnl2_carry"] = pnl2_carry
        df.at[t, "pnl2_yield"] = pnl2_yield
        df.at[t, "pnl10_carry"] = pnl10_carry
        df.at[t, "pnl10_yield"] = pnl10_yield
        df.at[t, "pnl_carry"] = pnl_carry
        df.at[t, "pnl_yield"] = pnl_yield
        df.at[t, "pnl"] = total_pnl
        df.at[t, "cash_interest"] = cash_int
        df.at[t, "y_2p"] = df.at[tprev, "y_2"]
        df.at[t, "y_10p"] = df.at[tprev, "y_10"]
    else:
        # First week: no prior positions or cash accrual.
        df.at[t, "pnl"] = 0.0
        df.at[t, "cash_interest"] = 0.0
        df.at[t, "ret"] = 0.0

    if is_last:
        # Final unwind: no new position is opened on the last observation.
        df.at[t, "q2"] = 0.0
        df.at[t, "q10"] = 0.0
        df.at[t, "mv2"] = 0.0
        df.at[t, "mv10"] = 0.0
        df.at[t, "gross_exposure"] = 0.0
        df.at[t, "longpos"] = 0.0
        df.at[t, "shortpos"] = 0.0
        df.at[t, "cash"] = equity
        df.at[t, "equity"] = equity
    else:
        hedge = r.DV01_10 / r.DV01_2
        max_gross = equity / margin_rate
        gross_unit = abs(hedge) * r.P2 + r.P10
        scale = max_gross / gross_unit

        q10 = scale
        q2 = -hedge * scale

        df.at[t, "q10"] = q10
        df.at[t, "q2"] = q2

        longpos = q10 * r.P10
        shortpos = -q2 * r.P2
        df.at[t, "longpos"] = longpos
        df.at[t, "shortpos"] = shortpos
        df.at[t, "mv10"] = q10 * r.P10
        df.at[t, "mv2"] = q2 * r.P2
        df.at[t, "gross_exposure"] = abs(q2) * r.P2 + abs(q10) * r.P10

        # Self-financing residual cash after re-hedging.
        cash = equity - (q2 * r.P2 + q10 * r.P10)
        df.at[t, "cash"] = cash
        df.at[t, "equity"] = equity

    equity_prev = equity


In [ ]:
# Cumulative P&L and returns.
df["pnl_positions"] = df["pnl_carry"].fillna(0) + df["pnl_yield"].fillna(0)
df["cum_pnl_positions"] = df["pnl_positions"].cumsum()
df["cum_cash_interest"] = df["cash_interest"].fillna(0).cumsum()
df["cum_pnl"] = df["pnl"].fillna(0).cumsum()
df["cum_return"] = (1.0 + df["ret"].fillna(0)).cumprod() - 1.0

ax = df["cum_return"].plot(
    figsize=(10, 4),
    title="Cumulative Return of DV01-Neutral 2s10s Flattener Strategy",
)
ax.set_ylabel("Cumulative return")
ax.set_xlabel("Date")
plt.tight_layout()
plt.show()


In [ ]:
# Convexity risk time series for a +10bp parallel shock.
bp_shock = 10e-4
face_10 = 1_000_000.0

# Second derivative of a zero-coupon price with respect to yield.
DV02_2 = (T2 ** 2) * df["P2"]
DV02_10 = (T10 ** 2) * df["P10"]

q10_face = face_10
q2_face = -q10_face * df["hedge_ratio"]

# Second-order price effect: 0.5 * P''(y) * (Δy)^2.
df["convexity_pnl_+10bp"] = 0.5 * (
    q10_face * DV02_10 + q2_face * DV02_2
) * (bp_shock ** 2)

ax = df["convexity_pnl_+10bp"].plot(
    figsize=(10, 4),
    title="Convexity Risk (DV01-Neutral 2s10s), +10bp Parallel Shift",
)
ax.set_ylabel("Convexity contribution (USD) for +10bp")
ax.set_xlabel("Date")
plt.tight_layout()
plt.show()


In [ ]:
# Approximate weekly P&L decomposition into spread, convexity, time/carry, and residual.
t0 = df.index[0]
for c in ["pnl_spread", "pnl_convex", "pnl_time", "pnl_resid"]:
    df.at[t0, c] = 0.0

for i, t in enumerate(df.index[1:], start=1):
    tprev = df.index[i - 1]
    rprev = df.loc[tprev]
    r = df.loc[t]

    q2_prev = df.at[tprev, "q2"]
    q10_prev = df.at[tprev, "q10"]
    cash_prev = df.at[tprev, "cash"]
    y1w_prev = df.at[tprev, "y_1w"]

    y2_prev, y10_prev = rprev.y_2, rprev.y_10
    y2_today, y10_today = r.y_2, r.y_10
    P2_prev, P10_prev = df.at[tprev, "P2"], df.at[tprev, "P10"]

    DV01_2_prev = df.at[tprev, "DV01_2"]
    DV01_10_prev = df.at[tprev, "DV01_10"]
    DV02_2_prev = (T2 ** 2) * P2_prev / 1e8
    DV02_10_prev = (T10 ** 2) * P10_prev / 1e8

    dy2 = y2_today - y2_prev
    dy10 = y10_today - y10_prev

    # First-order rate-move component.
    pnl_spread = -(
        q10_prev * DV01_10_prev * dy10 * 1e4
        + q2_prev * DV01_2_prev * dy2 * 1e4
    )

    # Second-order convexity component.
    pnl_convex = 0.5 * (
        q10_prev * DV02_10_prev * (dy10 * 1e4) ** 2
        + q2_prev * DV02_2_prev * (dy2 * 1e4) ** 2
    )

    # Time/carry component from rolling down last week's curve plus cash accrual.
    y2_prev_roll = y_from_prev_curve(rprev, T2 - dt)
    y10_prev_roll = y_from_prev_curve(rprev, T10 - dt)
    P2_carry = np.exp(-y2_prev_roll * (T2 - dt))
    P10_carry = np.exp(-y10_prev_roll * (T10 - dt))
    pnl_carry_time = (
        q2_prev * (P2_carry - P2_prev)
        + q10_prev * (P10_carry - P10_prev)
    )
    cash_int = cash_prev * (np.exp(y1w_prev * dt) - 1.0)
    pnl_time = pnl_carry_time + cash_int

    pnl_total = df.at[t, "pnl"]
    pnl_resid = pnl_total - (pnl_spread + pnl_convex + pnl_time)

    df.at[t, "pnl_spread"] = pnl_spread
    df.at[t, "pnl_convex"] = pnl_convex
    df.at[t, "pnl_time"] = pnl_time
    df.at[t, "pnl_resid"] = pnl_resid

for c in ["pnl", "pnl_spread", "pnl_convex", "pnl_time", "pnl_resid"]:
    df[f"cum_{c}"] = df[c].fillna(0).cumsum()

plt.figure(figsize=(10, 6))
plt.plot(df.index, df["cum_pnl"], label="Total")
plt.plot(df.index, df["cum_pnl_spread"], label="Spread / rate move")
plt.plot(df.index, df["cum_pnl_convex"], label="Convexity")
plt.plot(df.index, df["cum_pnl_time"], label="Time (carry + cash)")
plt.plot(df.index, df["cum_pnl_resid"], label="Residual")
plt.title("Cumulative P&L Decomposition (DV01-Neutral 2s10s)")
plt.ylabel("Cumulative P&L ($)")
plt.xlabel("Date")
plt.legend()
plt.tight_layout()
plt.show()


In [ ]:
# Compare leverage implied by 10% vs 2% margin requirements.
def run_margin(df0, margin):
    dfm = df0.copy()

    for c in ["q2", "q10", "cash", "ret", "cash_interest"]:
        dfm[c] = np.nan

    equity = equity_init
    equity_prev = equity_init
    idx = dfm.index.to_list()

    for i, t in enumerate(idx):
        r = dfm.loc[t]

        if i > 0:
            tprev = idx[i - 1]
            rprev = dfm.loc[tprev]
            q2_prev = dfm.at[tprev, "q2"]
            q10_prev = dfm.at[tprev, "q10"]

            P2_prev = dfm.at[tprev, "P2"]
            P10_prev = dfm.at[tprev, "P10"]

            y2_prev_roll = y_from_prev_curve(rprev, T2 - dt)
            y10_prev_roll = y_from_prev_curve(rprev, T10 - dt)
            P2_carry = np.exp(-y2_prev_roll * (T2 - dt))
            P10_carry = np.exp(-y10_prev_roll * (T10 - dt))

            y2_today_roll = curve_yield(r, T2 - dt)
            y10_today_roll = curve_yield(r, T10 - dt)
            P2_today = np.exp(-y2_today_roll * (T2 - dt))
            P10_today = np.exp(-y10_today_roll * (T10 - dt))

            pnl_carry = (
                q2_prev * (P2_carry - P2_prev)
                + q10_prev * (P10_carry - P10_prev)
            )
            pnl_yield = (
                q2_prev * (P2_today - P2_carry)
                + q10_prev * (P10_today - P10_carry)
            )

            cash_prev = dfm.at[tprev, "cash"]
            r1w_prev = dfm.at[tprev, "y_1w"]
            cash_int = cash_prev * (np.exp(r1w_prev * dt) - 1.0)

            pnl_total = pnl_carry + pnl_yield + cash_int
            equity += pnl_total
            dfm.at[t, "ret"] = pnl_total / equity_prev
            dfm.at[t, "cash_interest"] = cash_int
        else:
            dfm.at[t, "ret"] = 0.0
            dfm.at[t, "cash_interest"] = 0.0

        if i == len(idx) - 1:
            dfm.at[t, "q2"] = 0.0
            dfm.at[t, "q10"] = 0.0
            dfm.at[t, "cash"] = equity
        else:
            h = r.hedge_ratio
            gross_unit = abs(h) * r.P2 + r.P10
            q10 = (equity / margin) / gross_unit
            q2 = -h * q10

            dfm.at[t, "q10"] = q10
            dfm.at[t, "q2"] = q2
            dfm.at[t, "cash"] = equity - (q2 * r.P2 + q10 * r.P10)

        equity_prev = equity

    return (1.0 + dfm["ret"].fillna(0)).cumprod() - 1.0


cum10 = run_margin(df, 0.10)
cum02 = run_margin(df, 0.02)

ax = cum10.plot(figsize=(9, 4), label="10% margin")
cum02.plot(ax=ax, label="2% margin")
ax.set_title("Cumulative Return: 2% vs 10% Margin (DV01-Neutral 2s10s)")
ax.set_ylabel("Cumulative return")
ax.set_xlabel("Date")
ax.legend()
plt.tight_layout()
plt.show()
